In [ ]:
!pip install -U torch==2.5.1 'accelerate>=0.26.0' bitsandbytes qwen_vl_utils torchvision transformers huggingface_hub fbgemm-gpu

In [1]:
!nvidia-smi

Tue May 27 16:15:09 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.127.08             Driver Version: 550.127.08     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          On  |   00000000:26:00.0 Off |                    0 |
| N/A   41C    P0            151W /  700W |       1MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [1]:
from simulations_core import *

In [2]:
model_label = 'mistral_24B'

model, processor = import_mistral_24B()

generation_kwargs = {
    "max_new_tokens": 256,
    "do_sample": False,
    # "top_k": 50,
    # "top_p": 0.95,
    # "temperature": 0.15,
    "output_scores": True,               # <--- Aggiunto
    "return_dict_in_generate": True,      # <--- Aggiunto
    "pad_token_id": processor.tokenizer.eos_token_id  # 👈 aggiungi questa riga
}

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/10 [00:00<?, ?it/s]

## CREATE GENERAL POOL OF IMAGES

In [7]:
import os
import random
import shutil
from tqdm import tqdm
from PIL import Image
import numpy as np

# Parametri
n_images = 100
square_size = 200
spacing = 100
dpi = 150
labels = ['A', 'B']
bound = 50

# Directory di salvataggio
output_dir = f"images_color_{model_label}"
os.makedirs(output_dir, exist_ok=True)


# Generazione immagini
fig_index = 0
attempts = 0
pbar = tqdm(total=n_images, desc="Generazione immagini valide")

while fig_index < n_images:
    position = random.randint(0, 1)
    correct_answer = labels[position]

    # Estrai reference RGB casuale
    ref_rgb = [random.randint(0 + bound, 255 - bound) for _ in range(3)]
    reference_color = f"({ref_rgb[0]},{ref_rgb[1]},{ref_rgb[2]})"

    # Estrai other RGB entro i bound
    other_rgb = [random.randint(max(0, c - bound), min(255, c + bound)) for c in ref_rgb]
    other_color = f"({other_rgb[0]},{other_rgb[1]},{other_rgb[2]})"

    temp_path = os.path.join(output_dir, 'temp.png')

    # Crea immagine
    create_image_color(reference_color, other_color, position, output_dir, 'temp.png')

    prompt = (
        f"In the image, there are three colored squares labeled {labels[0]}, REFERENCE COLOR, and {labels[1]}.\n"
        f"Which of the squares, {labels[0]} or {labels[1]}, has the same color as the REFERENCE COLOR?\n"
        f"Provide only the final answer, either {labels[0]} or {labels[1]}, without generating anything else."
    )

    # Chiamata al modello
    prob_labels, output_scores = single_query_mistral(prompt, temp_path, labels, model, processor, generation_kwargs, print_flag=False)
    prob_correct = prob_labels.get(correct_answer, 0.0)

    image_name = f"colors_{fig_index}_{correct_answer}_p={prob_correct:.2f}.png"
    image_path = os.path.join(output_dir, image_name)

    # Verifica soglia
    if prob_correct >= 0.95:
        fig_index += 1
        pbar.update(1)
        os.rename(temp_path, image_path)
    else:
        os.remove(temp_path)

    attempts += 1

pbar.close()
print(f"\n✅ Completato: {n_images} immagini generate correttamente su {attempts} tentativi.")


Generazione immagini valide: 100%|██████████| 100/100 [00:30<00:00,  3.29it/s]


✅ Completato: 100 immagini generate correttamente su 140 tentativi.


## CREATE IMAGES FOR TASK DIFFICULTY AND PERFORMANCE

In [ ]:
import os
import random
import shutil
from tqdm import tqdm
from PIL import Image
import numpy as np

# Parametri
n_images = 50
square_size = 200
spacing = 100
dpi = 150
labels = ['A', 'B']
# bounds = [20]
bounds = range(10, 111, 10)

# Directory di salvataggio
output_dir = f"images_color_perplexity_bound_{model_label}"
os.makedirs(output_dir, exist_ok=True)

for bound in bounds:

    print(f'bound = {bound}')
    
# Generazione immagini
    fig_index = 0
    attempts = 0
    pbar = tqdm(total=n_images, desc="Generazione immagini valide")
    
    while fig_index < n_images:
        position = random.randint(0, 1)
        correct_answer = labels[position]
    
        # Estrai reference RGB casuale
        ref_rgb = [random.randint(0 + bound, 255 - bound) for _ in range(3)]
        reference_color = f"({ref_rgb[0]},{ref_rgb[1]},{ref_rgb[2]})"
    
        # Estrai other RGB entro i bound
        other_rgb = [random.randint(max(0, c - bound), min(255, c + bound)) for c in ref_rgb]
        other_color = f"({other_rgb[0]},{other_rgb[1]},{other_rgb[2]})"
    
        temp_path = os.path.join(output_dir, 'temp.png')
    
        # Crea immagine
        create_image_color(reference_color, other_color, position, output_dir, 'temp.png')
    
        prompt = (
            f"In the image, there are three colored squares labeled {labels[0]}, REFERENCE COLOR, and {labels[1]}.\n"
            f"Which of the squares, {labels[0]} or {labels[1]}, has the same color as the REFERENCE COLOR?\n"
            f"Provide only the final answer, either {labels[0]} or {labels[1]}, without generating anything else."
        )
    
        # Chiamata al modello
        prob_labels, output_scores = single_query_mistral(prompt, temp_path, labels, model, processor, generation_kwargs, print_flag=False)
        prob_correct = prob_labels.get(correct_answer, 0.0)
        logit = output_scores.get(correct_answer, 0.0)
    
        image_name = f"colors_{fig_index}_{correct_answer}_p={prob_correct:.2f}_logit={logit:.2f}_bound={bound}.png"
        image_path = os.path.join(output_dir, image_name)
    
        # Verifica soglia
        if prob_correct >= 0.95:
            fig_index += 1
            pbar.update(1)
            os.rename(temp_path, image_path)
        else:
            os.remove(temp_path)
    
        attempts += 1
    
    pbar.close()
    print(f"\n✅ Completato: {n_images} immagini generate correttamente su {attempts} tentativi.")


bound = 10


Generazione immagini valide: 100%|██████████| 50/50 [04:01<00:00,  4.83s/it]



✅ Completato: 50 immagini generate correttamente su 1163 tentativi.
bound = 20


Generazione immagini valide: 100%|██████████| 50/50 [00:37<00:00,  1.34it/s]



✅ Completato: 50 immagini generate correttamente su 188 tentativi.
bound = 30


...


✅ Completato: 50 immagini generate correttamente su 55 tentativi.
bound = 100


Generazione immagini valide: 100%|██████████| 50/50 [00:09<00:00,  5.04it/s]



✅ Completato: 50 immagini generate correttamente su 52 tentativi.
bound = 110


Generazione immagini valide: 100%|██████████| 50/50 [00:12<00:00,  4.16it/s]


✅ Completato: 50 immagini generate correttamente su 52 tentativi.
